# Three-dimensional rectangular particle in a box

**Prerequisite:** [Three-dimensional rectangular particle-in-a-box spectrum](../../../docs/lecture-notes/qm/piab3d/piab3d.md)

This maintained laboratory evaluates the continuum stationary-state energies of a particle confined by homogeneous Dirichlet boundaries on a rectangular cuboid. The selected examples are nondimensional, with $m=1$ and $\hbar=1$.

## Learning objectives

1. Enumerate states by explicit positive triples $(n_x,n_y,n_z)$.
2. Evaluate and independently check the analytical spectrum.
3. Identify permutation degeneracies in a cubic box.
4. Observe how unequal side lengths split permutation-related states.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab3d.base import Piab3D
from projectkoios.physkit.qm.piab3d.tise.analytical.solution import (
    Piab3DAnalyticalEvaluator,
)
from projectkoios.physkit.units import UnitSystem

In [ ]:
unit_box_length = 1.0
unit_particle_mass = 1.0
cubic_model = Piab3D(
    length_x=unit_box_length,
    length_y=unit_box_length,
    length_z=unit_box_length,
    mass=unit_particle_mass,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
maximum_quantum_number = 3
cubic_solution = Piab3DAnalyticalEvaluator().execute(
    model=cubic_model,
    maximum_quantum_number_x=maximum_quantum_number,
    maximum_quantum_number_y=maximum_quantum_number,
    maximum_quantum_number_z=maximum_quantum_number,
)

cubic_solution.quantum_numbers[:10], cubic_solution.energies.magnitude[:10]

In [ ]:
quantum_numbers = cubic_solution.quantum_numbers.astype(np.float64)
expected_energies = (
    np.pi**2
    / (2.0 * cubic_model.mass)
    * (
        quantum_numbers[:, 0] ** 2 / cubic_model.length_x**2
        + quantum_numbers[:, 1] ** 2 / cubic_model.length_y**2
        + quantum_numbers[:, 2] ** 2 / cubic_model.length_z**2
    )
)

assert np.allclose(
    cubic_solution.energies.magnitude,
    expected_energies,
    rtol=2e-15,
    atol=0.0,
)
assert np.all(np.diff(cubic_solution.energies.magnitude) >= 0.0)
assert np.isclose(
    cubic_solution.energies.magnitude[0],
    1.5 * np.pi**2,
    rtol=2e-15,
    atol=0.0,
)

In [ ]:
integer_mode_sums = np.sum(cubic_solution.quantum_numbers**2, axis=1)
unique_mode_sums, multiplicities = np.unique(
    integer_mode_sums,
    return_counts=True,
)
degenerate_levels = np.column_stack(
    (unique_mode_sums[multiplicities > 1], multiplicities[multiplicities > 1])
)

triple_to_energy = {
    tuple(triple): energy
    for triple, energy in zip(
        cubic_solution.quantum_numbers,
        cubic_solution.energies.magnitude,
        strict=True,
    )
}
assert triple_to_energy[(1, 1, 2)] == triple_to_energy[(1, 2, 1)]
assert triple_to_energy[(1, 2, 1)] == triple_to_energy[(2, 1, 1)]

degenerate_levels

In [ ]:
orthorhombic_length_x = 1.0
orthorhombic_length_y = 1.25
orthorhombic_length_z = 1.5
orthorhombic_maximum_quantum_number = 2
orthorhombic_model = Piab3D(
    length_x=orthorhombic_length_x,
    length_y=orthorhombic_length_y,
    length_z=orthorhombic_length_z,
    mass=unit_particle_mass,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
orthorhombic_solution = Piab3DAnalyticalEvaluator().execute(
    model=orthorhombic_model,
    maximum_quantum_number_x=orthorhombic_maximum_quantum_number,
    maximum_quantum_number_y=orthorhombic_maximum_quantum_number,
    maximum_quantum_number_z=orthorhombic_maximum_quantum_number,
)
orthorhombic_triple_to_energy = {
    tuple(triple): energy
    for triple, energy in zip(
        orthorhombic_solution.quantum_numbers,
        orthorhombic_solution.energies.magnitude,
        strict=True,
    )
}

permuted_energies = np.array(
    [
        orthorhombic_triple_to_energy[(1, 1, 2)],
        orthorhombic_triple_to_energy[(1, 2, 1)],
        orthorhombic_triple_to_energy[(2, 1, 1)],
    ]
)
assert np.unique(permuted_energies).size == 3
permuted_energies

In [ ]:
state_indices = np.arange(
    1,
    cubic_solution.energies.magnitude.size + 1,
    dtype=np.int64,
)
labels = [rf"$({n_x},{n_y},{n_z})$" for n_x, n_y, n_z in cubic_solution.quantum_numbers]

figure = plt.figure(figsize=(13, 5))
spectrum_axis = figure.add_subplot(1, 2, 1)
inventory_axis = figure.add_subplot(1, 2, 2, projection="3d")

spectrum_axis.scatter(state_indices, cubic_solution.energies.magnitude)
for state_index, energy, label in zip(
    state_indices[:10],
    cubic_solution.energies.magnitude[:10],
    labels[:10],
    strict=True,
):
    spectrum_axis.annotate(
        label,
        (state_index, energy),
        xytext=(3, 5),
        textcoords="offset points",
    )
spectrum_axis.set(
    xlabel="energy-ordered state index",
    ylabel="energy",
    title="First 27 states of the nondimensional cube",
)
spectrum_axis.grid(True)

inventory_plot = inventory_axis.scatter(
    cubic_solution.quantum_numbers[:, 0],
    cubic_solution.quantum_numbers[:, 1],
    cubic_solution.quantum_numbers[:, 2],
    c=cubic_solution.energies.magnitude,
    cmap="viridis",
)
inventory_axis.set(
    xlabel="$n_x$",
    ylabel="$n_y$",
    zlabel="$n_z$",
    title="Quantum-number inventory colored by energy",
)
figure.colorbar(inventory_plot, ax=inventory_axis, label="energy", shrink=0.75)

figure.tight_layout()
plt.show()

## Interpretation

The evaluator orders the finite state inventory by energy and then lexicographically resolves exact ties. A cube is invariant under permutations of the coordinate axes, so triples related by coordinate permutation have equal energy. Distinct side lengths remove this permutation symmetry and split those states.

This is an analytical continuum result for a homogeneous-Dirichlet cuboid. The three-dimensional scatter plot represents the finite quantum-number inventory, not physical coordinate space.

## Exercises

1. Increase all maximum quantum numbers and identify energy levels with the largest multiplicity.
2. Double all side lengths and verify inverse-square energy scaling.
3. Hold $L_x$ and $L_y$ fixed while varying $L_z$ and track permutation-related states.
4. Derive the normalized product eigenfunctions and visualize planar probability-density slices.
